# Phase 3 · Contrastive refinement (DDP)
Continues training the phase-2 adapter (`runs/<mode>/align/final`: LoRA r=16 + head, base frozen) against the **frozen teacher document vectors** (`corpus.npy`, never re-encoded):

$$\mathcal L = -\frac1B\sum_i \log\frac{e^{s(q_i,d_i^+)/\tau}}{e^{s(q_i,d_i^+)/\tau}+\sum_{u\in\mathcal N_i}e^{s_u/\tau}} \;+\; \lambda\,\lVert f_Q^S(q_i)-f_Q^T(q_i)\rVert^2,\qquad s(q,d)=\langle f_Q^S(q),f_D^T(d)\rangle$$

- **Training queries:** the `train` split (only split with positives in the corpus). One positive per query per step (rotating over its positives by epoch).
- **$\mathcal N_i$:** every other doc of the global batch (all queries' positives + their stored `negative[0]`, which is *not verified* BM25) + the other student queries on the GPU (same tower).
- **False negatives masked:** $q_i$'s other labelled positives, and any negative with $s_u > s(q_i,d_i^+) + \text{margin}$.
- **$\lambda$ (`l2_weight`):** keeps the student near the teacher query space (teacher vectors from `align.npy`); 0 = pure contrastive.

Checkpoints are adapter-only, as in phase 2: `runs/<mode>/refine/step_<n>/` (resumable) and `runs/<mode>/refine/final/`. The phase-2 adapter is read, never modified.

In [ ]:
# setup: clone/pull the repo on Kaggle, put it on sys.path, install what's missing (re-run to pull latest code)
import os, sys, importlib.util
REPO_URL = "https://github.com/pyther-hub/hlm-poc.git"
ON_KAGGLE = os.path.exists("/kaggle/working")
REPO = "/kaggle/working/hlm-poc" if ON_KAGGLE else os.path.abspath("..")
if ON_KAGGLE:
    if not os.path.exists(REPO):
        !git clone -q {REPO_URL} {REPO}
    else:
        !git -C {REPO} pull -q
sys.path.insert(0, REPO)

# HF token (higher rate limits, faster downloads). Kaggle: Add-ons -> Secrets -> add "HF_TOKEN", attach to notebook.
# Spawned workers inherit os.environ. Locally: export HF_TOKEN or `huggingface-cli login`.
if ON_KAGGLE and "HF_TOKEN" not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print(f"HF_TOKEN secret not found ({type(e).__name__}): downloads stay unauthenticated")

import transformers
from packaging.version import Version
if Version(transformers.__version__) < Version("4.56"):  # Qwen3 + the dtype= kwarg
    !pip install -q -U "transformers>=4.56"
    print("transformers upgraded: restart the kernel, then re-run this cell")
for pkg in ("datasets", "pyarrow", "accelerate"):
    if importlib.util.find_spec(pkg) is None:
        !pip install -q {pkg}

%load_ext autoreload
%autoreload 2

## Control panel

In [ ]:
import torch
from teacher import TEACHER_DEFAULTS as T

DEBUG = True    # refines on the debug cache + debug phase-2 adapter. Set False for the full run.
NPROCS = 2      # one process per T4

# --- paths: teacher cache + phase-2 adapter in, refined adapter out; final metrics go next to the phase_1_eval rows
ROOT = "/kaggle/working" if ON_KAGGLE else REPO
MODE = "debug" if DEBUG else "full"
DATA_PATH = f"{ROOT}/cache/{MODE}/data.json"
EMB_DIR = f"{ROOT}/cache/{MODE}/emb/teacher"
INIT_ADAPTER = f"{ROOT}/runs/{MODE}/align/final"
OUT_DIR = f"{ROOT}/runs/{MODE}/refine"
EVAL_DIR = f"{ROOT}/runs/{MODE}/eval"

# --- refinement (refine.REFINE_DEFAULTS; unknown keys raise)
REFINE = dict(init_adapter=INIT_ADAPTER,
              dim=T["dim"],                        # = adapter head output = teacher index dim
              query_prefix="", max_query_len=32,   # same as phase 2
              tau=0.05,                            # temperature
              margin=0.1,                          # mask negatives with s > s(q, d+) + margin
              n_neg=1,                             # negatives per query: 1 = stored one, k > 1 adds k-1 teacher-mined
              l2_weight=0.1,                       # lambda for L2(student q, teacher q); 0 = pure L_CR
              lr=5e-5, head_lr=1e-4, weight_decay=0.0, warmup_ratio=0.05,
              epochs=1, batch_size=64,             # queries per GPU; docs per step = (1 + n_neg) * batch_size * NPROCS
              grad_clip=1.0, amp=True, seed=0,
              log_every=20, eval_every=200,
              save_every=500, keep_last=2,
              data_path=DATA_PATH, teacher_emb_dir=EMB_DIR, out_dir=OUT_DIR)
if DEBUG:
    REFINE.update(log_every=5, eval_every=20, save_every=20)

## 1 · Check inputs
The phase-2 adapter exists and matches the teacher dim; teacher vectors for corpus, align (covers the train queries) and val are valid.

In [ ]:
from data import load_emb, load_json
from evaluation import check_emb

cfg = load_json(f"{INIT_ADAPTER}/adapter_config.json")   # written by phase_2_align
print("phase-2 adapter:", cfg)
assert cfg["out_dim"] == REFINE["dim"], f"adapter dim {cfg['out_dim']} != {REFINE['dim']}"
data = load_json(DATA_PATH)
for name in ("corpus", "align", "val"):
    emb, ids = load_emb(EMB_DIR, name)
    check_emb(emb, ids, data[name]["ids"], name)
    print(f"{name:<7} {emb.shape}")
assert set(data["train"]["ids"]) <= set(data["align"]["ids"]), "train queries missing from align"
per_step = REFINE["batch_size"] * NPROCS
print(f"{len(data['train']['ids'])} train queries, global batch {per_step} queries + {2 * per_step} docs, "
      f"{len(data['train']['ids']) // per_step * REFINE['epochs']} steps")

## 2 · Train
Resumable: re-running continues from the latest `step_<n>/`. A no-op once `final/` exists; delete `OUT_DIR` to train again.

In [ ]:
from ddp import launch
from refine import refine_main

launch(refine_main, REFINE, nprocs=NPROCS)

## 3 · Training log
`cr` = contrastive term, `l2` = alignment term, `masked` = fraction of negatives masked as false negatives (if it is large, `margin` is too tight).

In [ ]:
from align import read_log

log = read_log(OUT_DIR)
for r in log[::max(1, len(log) // 30)]:
    if "train_loss" in r:
        print(f"step {r['step']:>6}  loss {r['train_loss']:.4f}  cr {r['cr']:.4f}  l2 {r['l2']:.4f}  "
              f"masked {r['masked']:.3f}  lr {r['lr']:.2e}")
for r in log:
    if "val_loss" in r:
        print(f"step {r['step']:>6}  val_l2 {r['val_loss']:.4f}  val_cos {r['val_cos']:.4f}  "
              f"ndcg@10 {r['val_ndcg@10']:.4f}  recall@10 {r['val_recall@10']:.4f}")

## 4 · Evaluate the refined adapter
Same scoring as `phase_1_eval`, merged into its `metrics.json` next to the phase-2 row. Retention needs the `teacher` row (`phase_1_eval` on the same cache).

In [ ]:
from evaluation import evaluate_model, load_index, save_metrics, summarize
from model import get_device

device = get_device()
docs, _ = load_index(data, EMB_DIR, ("val", "test"))
name = f"refined_tau{REFINE['tau']}_l2w{REFINE['l2_weight']}"
result = evaluate_model(f"{OUT_DIR}/final", data, docs, splits=("val", "test"), dim=REFINE["dim"],
                        max_query_len=REFINE["max_query_len"], query_prefix=REFINE["query_prefix"], k=100,
                        device=device, dtype="fp16")
save_metrics(EVAL_DIR, {name: result})
for split in ("val", "test"):
    summarize(EVAL_DIR, split)
    print()